# 📊 Zone Response Analysis
**Purpose:** Analyse how zone temperature, humidity, and CO₂ respond to VAV flow commands.  
Select a single zone at the top and all downstream plots update automatically.

In [65]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from data_wrangling import DataInspector, DataPlotter

# Load data
CSV_PATH = './results/state_log.csv'
df = pd.read_csv(CSV_PATH)

# Build datetime index (EnergyPlus uses Hour=24 for midnight → roll to next day)
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)

ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']

print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')
# df.head(3)


Loaded 288 timesteps, columns: 127


In [66]:
ZONE = 'SPACE1-1'

## 1 · Temperature Response vs Flow Command

In [67]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Zone & Outdoor Temperature', 'VAV Flow Command'],
    vertical_spacing=0.08
)

# Row 1: Temperatures
fig.add_trace(go.Scatter(x=df['Datetime'], y=df['Out_Temp_C'],
    name='Outdoor T', line=dict(color='#636EFA', width=1, dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'Fan_Out_Temp_C'],
    name='Supply Air T', line=dict(color='#00CC96', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_Temp_C'],
    name=f'{ZONE} T_in', line=dict(color='#EF553B', width=2)), row=1, col=1)
# fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_T_m_C'],
#     name=f'{ZONE} T_m (Radiant)', line=dict(color='#FFA15A', width=1.5)), row=1, col=1)

# Row 2: VAV flow
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_VAV_Flow_kg_s'],
    name='VAV Flow', fill='tozeroy',
    line=dict(color='#AB63FA', width=1.5)), row=2, col=1)

fig.update_layout(template='plotly_dark', height=550,
    title=f'Temperature Response — {ZONE}',
    legend=dict(orientation='h', y=-0.12))
fig.update_yaxes(title_text='°C', row=1, col=1)
fig.update_yaxes(title_text='kg/s', row=2, col=1)
fig.show()

/home/jazz/.local/lib/python3.14/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## 2 · Humidity Response vs Flow Command

In [68]:
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['Zone Humidity (RH & W_in)', 'VAV Flow Command'],
    specs=[[{'secondary_y': True}], [{}]],
    vertical_spacing=0.08
)

# RH on primary y-axis
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_RH_pct'],
    name=f'{ZONE} RH %', line=dict(color='#19D3F3', width=2)), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df['Out_RH_pct'],
    name='Outdoor RH %', line=dict(color='#636EFA', width=1, dash='dot')), row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df['Fan_Out_RH_pct'],
    name='Supply RH %', line=dict(color='#FFA15A', width=1, dash='dot')), row=1, col=1, secondary_y=False)

# W_in on secondary y-axis
# fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_W_in_kg_kg'],
#     name=f'{ZONE} W_in (kg/kg)', line=dict(color='#FFA15A', width=1.5)), row=1, col=1, secondary_y=True)

# Row 2: Flow
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_VAV_Flow_kg_s'],
    name='VAV Flow', fill='tozeroy',
    line=dict(color='#AB63FA', width=1.5)), row=2, col=1)

fig.update_layout(template='plotly_dark', height=550,
    title=f'Humidity Response — {ZONE}',
    legend=dict(orientation='h', y=-0.12))
fig.update_yaxes(title_text='RH %', row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text='kg/kg', row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text='kg/s', row=2, col=1)
fig.show()

## 3 · CO₂ Response vs Flow & Occupancy

In [69]:
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.45, 0.30, 0.25],
    subplot_titles=['Zone CO₂ Concentration', 'VAV Flow Command', 'Occupancy'],
    vertical_spacing=0.07
)

fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_CO2_ppm'],
    name=f'{ZONE} CO₂', line=dict(color='#B6E880', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'Fan_Out_CO2_ppm'],
    name='Supply CO₂', line=dict(color='#00CC96', width=1, dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'Outdoor_Air_CO2_ppm'],
    name='Outside CO₂', line=dict(color='#AB63FA', width=1, dash='dot')), row=1, col=1)

fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_VAV_Flow_kg_s'],
    name='VAV Flow', fill='tozeroy',
    line=dict(color='#AB63FA', width=1.5)), row=2, col=1)

fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{ZONE}_Occupants'],
    name='Occupants', fill='tozeroy',
    line=dict(color='#FF6692', width=1.5)), row=3, col=1)

fig.update_layout(template='plotly_dark', height=650,
    title=f'CO₂ Response — {ZONE}',
    legend=dict(orientation='h', y=-0.08))
fig.update_yaxes(title_text='ppm', row=1, col=1)
fig.update_yaxes(title_text='kg/s', row=2, col=1)
fig.update_yaxes(title_text='Count', row=3, col=1)
fig.show()

## 4 · Supply Air Conditions (Outdoor → Fan Out)

In [70]:
fig = make_subplots(
    rows=2, cols=2, shared_xaxes=True,
    subplot_titles=['AHU Temperature Path', 'AHU RH Path',
                    'AHU Flow Path', 'AHU CO₂ Path'],
    vertical_spacing=0.12, horizontal_spacing=0.08
)

nodes = ['Outdoor_Air', 'Mixed_Air', 'CC_Out', 'HC_Out', 'Fan_Out']
colors_ahu = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA']

for i, node in enumerate(nodes):
    c = colors_ahu[i]
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_Temp_C'],
        name=f'{node} T', line=dict(color=c, width=1.5), legendgroup=node), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_RH_pct'],
        name=f'{node} RH', line=dict(color=c, width=1.5), legendgroup=node,
        showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_Flow_kg_s'],
        name=f'{node} Flow', line=dict(color=c, width=1.5), legendgroup=node,
        showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=df['Datetime'], y=df[f'{node}_CO2_ppm'],
        name=f'{node} CO₂', line=dict(color=c, width=1.5), legendgroup=node,
        showlegend=False), row=2, col=2)

fig.update_layout(template='plotly_dark', height=600,
    title='Air Handling Unit — State Through Nodes',
    legend=dict(orientation='h', y=-0.08))
fig.show()

## 5 · Statistical Summary (data_wrangling)

In [71]:
# Use DataInspector from the data_wrangling library
zone_cols = [f'{ZONE}_Temp_C', f'{ZONE}_RH_pct', f'{ZONE}_CO2_ppm',
             f'{ZONE}_VAV_Flow_kg_s', f'{ZONE}_Occupants', f'{ZONE}_EquipLoad_W',
             'Out_Temp_C', 'Out_RH_pct']

di = DataInspector()
di.df = df[zone_cols].copy()
sum_df= di.get_summary(detailed=True)

--- Data Summary (Detailed) ---


,Column Name,Data Type,Total Records,Missing Values,Min,Max,Mean,Std Dev,Unique Values,Most Frequent
0,SPACE1-1_Temp_C,float64,288,0,20.96,29.7300,24.190278,1.959873,NaN,NaN
1,SPACE1-1_RH_pct,float64,288,0,29.42,62.5200,48.939583,8.696362,NaN,NaN
2,SPACE1-1_CO2_ppm,float64,288,0,466.57,1269.4800,603.571181,154.309813,NaN,NaN
3,SPACE1-1_VAV_Flow_kg_s,float64,288,0,0.00,1.0881,0.177149,0.198952,NaN,NaN
4,SPACE1-1_Occupants,float64,288,0,0.00,38.0000,5.569444,10.235657,NaN,NaN
5,SPACE1-1_EquipLoad_W,float64,288,0,429.60,3818.4000,1119.823333,886.259447,NaN,NaN
6,Out_Temp_C,float64,288,0,11.70,24.4000,18.488646,3.301103,NaN,NaN
7,Out_RH_pct,float64,288,0,38.00,100.0000,76.890625,17.205302,NaN,NaN


In [72]:
# Flow vs Temperature scatter using DataPlotter
DataPlotter.scatter(
    df, x_col=f'{ZONE}_VAV_Flow_kg_s', y_col=f'{ZONE}_Temp_C',
    color_col=f'{ZONE}_Occupants',
    title=f'Flow vs Temperature — {ZONE}'
).show()